<a href="https://colab.research.google.com/github/ColumbiaPlus/GENAI_BizAnalytics/blob/main/Basics_of_text_analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization
from tensorflow.keras.layers import Embedding


**Tensorflow TextVectorization**
* The TextVectorization module maps text features to integers
* Details: https://www.tensorflow.org/api_docs/python/tf/keras/layers/TextVectorization


**Example**
* A corpus of four sentences

In [3]:
#The corpus

s1 = "Jack, a patriot, went to Boston to buy a banana"
s2 = "Jill likes to take a banana to Boston on a train"
s3 = "Habits are hard to break"
s4 = "Jack, as a patriotic American, took his flag to Boston"

corpus = [s1,s2,s3,s4]


#TextVectorization set up
vectorizer = TextVectorization(standardize='lower_and_strip_punctuation', #separates, for ex, Jack and ,
                               split='whitespace', #Uses spaces to split tokens
                               ngrams=None, #No combinations (simple model)
                               output_mode="count")

#Create the vocabulary

vectorizer.adapt(corpus)



**Vocabulary**
* Note that the comma is also a token
* The vocabulary contains each word in the corpus

In [4]:
#print length of vocabulary
print("Vocabulary size: ",vectorizer.vocabulary_size())

print(vectorizer.get_vocabulary())

Vocabulary size:  24
['[UNK]', 'to', 'a', 'boston', 'jack', 'banana', 'went', 'train', 'took', 'take', 'patriotic', 'patriot', 'on', 'likes', 'jill', 'his', 'hard', 'habits', 'flag', 'buy', 'break', 'as', 'are', 'american']


**Bag of Words**
* Build bag of word vectors for each sentence
* Note that the length of each vector is 24
* Word frequencies are at each location
* That there are a lot of 0's
* Try to imagine what a vector would look like if the vocabulary size was in the 1000000s!

In [5]:
w_v_1 = vectorizer(s1)
w_v_2 = vectorizer(s2)
w_v_3 = vectorizer(s3)
w_v_4 = vectorizer(s4)
print(w_v_1)
print(w_v_2)
print(w_v_3)
print(w_v_4)

tf.Tensor([0. 2. 2. 1. 1. 1. 1. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.], shape=(24,), dtype=float32)
tf.Tensor([0. 2. 2. 1. 0. 1. 0. 1. 0. 1. 0. 0. 1. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.], shape=(24,), dtype=float32)
tf.Tensor([0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 1. 0. 0. 1. 0. 1. 0.], shape=(24,), dtype=float32)
tf.Tensor([0. 1. 1. 1. 1. 0. 0. 0. 1. 0. 1. 0. 0. 0. 0. 1. 0. 0. 1. 0. 0. 1. 0. 1.], shape=(24,), dtype=float32)


**Embeddings**
* Embeddings are handled differently
* The vocabulary stays the same
* But both the ordering of words in sentences as well as the relationships between words can be considered
* And the total size of the data can be reduced by making word vectors of a limited size

In [7]:
#The corpus

s1 = "Jack, a patriot, went to Boston to buy a banana"
s2 = "Jill likes to take a banana to Boston on a train"
s3 = "Habits are hard to break"
s4 = "Jack, as a patriotic American, took his flag to Boston"

corpus = [s1,s2,s3,s4]


#TextVectorization set up
vectorizer = TextVectorization(standardize='lower_and_strip_punctuation', #separates, for ex, Jack and ,
                               split='whitespace', #Uses spaces to split tokens
                               ngrams=None, #No combinations (simple model)
                               output_mode="int",
                               output_sequence_length=5) #Usually, this is set to the largest sentence
                               #but, since we have a small corpus, I'm restricting this

#Create the vocabulary

vectorizer.adapt(corpus)



In [8]:
#print length of vocabulary. This doesn't change
print("Vocabulary size: ",vectorizer.vocabulary_size())

print(vectorizer.get_vocabulary())

Vocabulary size:  25
['', '[UNK]', 'to', 'a', 'boston', 'jack', 'banana', 'went', 'train', 'took', 'take', 'patriotic', 'patriot', 'on', 'likes', 'jill', 'his', 'hard', 'habits', 'flag', 'buy', 'break', 'as', 'are', 'american']


**Create embeddings**
*  Each sentence is represented by 5 vectors (the output_sequence_length we specified)
* Each of the 5 vectors represents a word using 4 floating point numbers
* The numbers are loadings and the similarity of one word vector with another represents similarity of usage
* But, the meanings are lost to us!


* The shape of the embeddings is (4,5,4)
* The first 4 is the number of sentences
* The 5 is the number of vectors used to represent the sentence. Typically, this is the length of the largest sentence
* The final 4 is the number of embeddings that represent each "concept" in the sentence

* The input size is a lot smaller
* Suppose
** 1000 sentences
** 1,000,000 vocabulary size
* Input size with bag of words = 1000 * 1,000,000 = 1,000,000,000
* Input size with embeddings = 1000 * 20 * 40 = 800,000  (1/1250 of the size) assuming
** longest sentence is 20 words
** embedding vector is of lenght 40
* Additional benefit: seqencing and relationships between words


In [14]:
vocab_len = vectorizer.vocabulary_size()
embed = Embedding(input_dim = vocab_len, output_dim = 4, input_length=4)
e = embed(vectorizer([s1,s2,s3,s4]))
print(e.shape)
print(e)

(4, 5, 4)
tf.Tensor(
[[[ 0.00547572  0.04511153  0.00296749  0.00682487]
  [-0.00950211  0.04303547  0.02645166 -0.04240357]
  [-0.04648007  0.02894386 -0.03866385 -0.00806166]
  [ 0.00178463 -0.03256246  0.04868397 -0.02464469]
  [ 0.04300657 -0.03477927  0.0426558  -0.03129053]]

 [[-0.03267368  0.03073463  0.01456424  0.01562685]
  [-0.01898313  0.03961729  0.04016234  0.01543358]
  [ 0.04300657 -0.03477927  0.0426558  -0.03129053]
  [ 0.00539436  0.03777869 -0.02063238 -0.02525829]
  [-0.00950211  0.04303547  0.02645166 -0.04240357]]

 [[ 0.03228008 -0.04579762  0.02942764  0.00438584]
  [ 0.04206326  0.00187729 -0.03660505  0.03752295]
  [ 0.03387168 -0.02480857  0.00056235  0.01492755]
  [ 0.04300657 -0.03477927  0.0426558  -0.03129053]
  [-0.04313191 -0.03921213 -0.03879211 -0.00531789]]

 [[ 0.00547572  0.04511153  0.00296749  0.00682487]
  [-0.02155896  0.00225232  0.02206982  0.0389944 ]
  [-0.00950211  0.04303547  0.02645166 -0.04240357]
  [-0.01219784 -0.02634384 -0.026115 